# XAI Metrics Summary and Method Comparison

Read all available `quant_metrics.json` files under the XAI output directory, aggregate the metrics into a single table, and export one consolidated CSV for manual comparison.

## 1. Configuration and Dependencies

In [ ]:
import json
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'dataset' / 'archive').exists() and (p / 'outputs').exists():
            return p
    raise FileNotFoundError('Could not locate the project root.')


PROJECT_ROOT = find_project_root(Path.cwd())
BASE_DIR = PROJECT_ROOT / 'outputs' / 'xai'
if not BASE_DIR.exists():
    raise FileNotFoundError(f'XAI output directory not found: {BASE_DIR}')
OUT_DIR = BASE_DIR / 'summary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional filters. Set to None to keep everything.
BASELINE_FILTER = None  # e.g. 'swin_t'
VARIANT_FILTER = None  # e.g. 'post-hoc' / 'intrinsic' / 'learned' / 'end-to-end'
METHOD_FILTER = None  # e.g. {'lime', 'shap'}

print('[INFO] PROJECT_ROOT   :', PROJECT_ROOT)
print('[INFO] XAI BASE DIR   :', BASE_DIR)
print('[INFO] SUMMARY OUT DIR:', OUT_DIR)

## 2. Utility Functions

In [ ]:
def discover_quant_files(base_dir: Path) -> list[Path]:
    return sorted(p for p in base_dir.rglob('quant_metrics.json') if 'summary' not in p.parts)


def load_quant_metrics(metrics_path: Path) -> dict[str, Any]:
    data = json.loads(metrics_path.read_text(encoding='utf-8'))
    metrics = data.get('metrics', {})
    result_dir = metrics_path.parent

    row = {
        'method': data.get('method'),
        'variant': data.get('variant'),
        'baseline_key': data.get('baseline_key'),
        'num_samples': data.get('num_samples'),
        'mean_explain_time_sec': metrics.get('mean_explain_time_sec'),
        'median_explain_time_sec': metrics.get('median_explain_time_sec'),
        'mean_aopc_deletion': metrics.get('mean_aopc_deletion'),
        'mean_auc_insertion': metrics.get('mean_auc_insertion'),
        'mean_iou_top10': metrics.get('mean_iou_top10'),
        'mean_comprehensiveness': metrics.get('mean_comprehensiveness'),
        'mean_sufficiency': metrics.get('mean_sufficiency'),
        'mean_selected_feature_ratio': metrics.get('mean_selected_feature_ratio'),
        'mean_sparsity': metrics.get('mean_sparsity'),
        'metrics_path': str(metrics_path),
        'per_sample_path': str(result_dir / 'per_sample_metrics.csv') if (result_dir / 'per_sample_metrics.csv').exists() else None,
        'visuals_dir': str(result_dir / 'visuals') if (result_dir / 'visuals').exists() else None,
    }
    row['xai_key'] = f"{row['variant']}::{row['method']}::{row['baseline_key']}"
    return row


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if BASELINE_FILTER is not None:
        out = out[out['baseline_key'] == BASELINE_FILTER]
    if VARIANT_FILTER is not None:
        out = out[out['variant'] == VARIANT_FILTER]
    if METHOD_FILTER is not None:
        out = out[out['method'].isin(set(METHOD_FILTER))]
    return out.reset_index(drop=True)

## 3. Aggregation and CSV Export

In [ ]:
quant_files = discover_quant_files(BASE_DIR)
if len(quant_files) == 0:
    print('[WARN] No quant_metrics.json files were found. Run one or more XAI notebooks first.')
else:
    rows = [load_quant_metrics(p) for p in quant_files]
    df = pd.DataFrame(rows)
    df = apply_filters(df)

    if len(df) == 0:
        print('[WARN] No XAI entries remained after applying the current filters.')
    else:
        sample_sizes = sorted(df['num_samples'].dropna().unique().tolist())
        if len(sample_sizes) > 1:
            print('[WARN] Methods were evaluated on different sample counts:', sample_sizes)

        display(df)

        all_csv = OUT_DIR / 'xai_metrics_all.csv'
        df.to_csv(all_csv, index=False, encoding='utf-8')

        print('[OK] XAI metrics summary:', all_csv)